# Project Code

## Installing

In [4]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install elasticsearch

Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install elasticsearch_dsl

Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install torch transformers

Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


## Load Dictionaries With Data

In [1]:
def read_qrels_file(qrels_file):  # reads the content of he qrels file
    trec_relevant = dict()  # query_id -> set([docid1, docid2, ...])
    with open(qrels_file, 'r') as qrels:
        for line in qrels:
            (qid, q0, doc_id, rel) = line.strip().split()
            if qid not in trec_relevant:
                trec_relevant[qid] = set()
            if (rel == "1"):
                trec_relevant[qid].add(doc_id)
    return trec_relevant

def read_run_file(run_file):  
    # read the content of the run file produced by our IR system 
    # (in the following exercises you will create your own run_files)
    trec_retrieved = dict()  # query_id -> [docid1, docid2, ...]
    with open(run_file, 'r') as run:
        for line in run:
            (qid, q0, doc_id, rank, score, tag) = line.strip().split()
            if qid not in trec_retrieved:
                trec_retrieved[qid] = []
            trec_retrieved[qid].append(doc_id) 
    return trec_retrieved
    

def read_eval_files(qrels_file, run_file):
    return read_qrels_file(qrels_file), read_run_file(run_file)

(all_relevant, all_retrieved) = read_eval_files('data01/FIR-s05-training-qrels.txt', 'data01/baseline.run')

## Connect To ElasticSearch

In [2]:
import elasticsearch
import elasticsearch.helpers
import json

def read_documents(file_name):
    """
    Returns a generator of documents to be indexed by elastic, read from file_name
    """
    with open(file_name, 'r') as documents:
        for line in documents:
            doc_line = json.loads(line)
            if ('index' in doc_line):
                id = doc_line['index']['_id']
            elif ('PMID' in doc_line):
                doc_line['_id'] = id
                yield doc_line
            else:
                raise ValueError('Woops, error in index file')

def create_index(es, index_name, body={}):
    # delete index when it already exists
    es.indices.delete(index=index_name, ignore=[400, 404])
    # create the index 
    es.indices.create(index=index_name, body=body)
                
def index_documents(es, collection_file_name, index_name, body={}):
    create_index(es, index_name, body)
    # bulk index the documents from file_name
    return elasticsearch.helpers.bulk(
        es, 
        read_documents(collection_file_name),
        index=index_name,
        chunk_size=2000,
        request_timeout=30
    )

In [3]:
from dotenv import load_dotenv
import os

# Load environment variables from the .env file to use elastic password
load_dotenv()

True

In [4]:
# Connect to the ElasticSearch server
# es = elasticsearch.Elasticsearch("https://localhost:9200/")

es = elasticsearch.Elasticsearch(
    "https://localhost:9200",
    basic_auth=("elastic", os.getenv("ELASTIC_PASSWORD")), 
    verify_certs=True,
    ca_certs="http_ca.crt"
)

## Create Index

In [5]:
# Index the collection into the index called 'genomics'
body = {} # no indexing options (leave default)
index_documents(es, 'data01/FIR-s05-medline.json', 'genomics-base', body)

C:\Users\daans\AppData\Local\Temp\ipykernel_25144\1573036711.py:22: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es.indices.delete(index=index_name, ignore=[400, 404])
C:\Users\daans\AppData\Local\Temp\ipykernel_25144\1573036711.py:29: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  return elasticsearch.helpers.bulk(


(263080, [])

## Create Run File On Queries

In [6]:
import elasticsearch_dsl

def make_trec_run_es(es, topics_file_name, run_file_name, index_name="genomics", run_name="test"):
    with open(run_file_name, 'w') as run_file:
        with open(topics_file_name, 'r') as test_queries:
            for line in test_queries:
                (qid, query) = line.strip().split('\t')
                
                s = elasticsearch_dsl.Search(using=es, index=index_name).query('multi_match', query=query, fields=['TI', 'AB']).extra(size=1000)
                                             
                response = s.execute()
                
                for rank, hit in enumerate(response.hits.hits[:1000]):
                    pmid = hit['_source']['PMID']
                    score = hit['_score']
                    
                    run_file.write(f"{qid} Q0 {pmid} {rank+1} {score} {run_name}\n")
                
make_trec_run_es(es, 'data01/FIR-s05-training-queries-simple.txt', 'result/elasticsearch.run', "genomics-base", run_name='project01')

## Bert ranking and retrieval

In [14]:
# make bert embeddings and write to a file

In [11]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
     ---------------------------------------- 6.1/6.1 MB 2.4 MB/s eta 0:00:00
     ---------------------------------------- 4.1/4.1 MB 2.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import json
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

# Suppress warning about symlink caching
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# Initialize DistilBERT components
tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext")
model = AutoModel.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext")

# Move the model to GPU if available and enable mixed precision
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()  # Set the model to evaluation mode


def get_bert_embeddings(texts):
    inputs = tokenizer(texts, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings

def prepare_text(document):
    """
    Prepare text for embedding by concatenating all relevant fields in the document.
    """
    fields_to_include = ["TI", "AB", "AD", "MH", "FAU", "SO"]
    text_parts = []

    for field in fields_to_include:
        content = document.get(field, "")
        if isinstance(content, list):
            content = " ".join(content)  # Join list items with a space
        text_parts.append(str(content))  # Ensure everything is a string

    return " ".join(text_parts)  # Concatenate with space between fields

def load_processed_ids(output_file):
    """
    Load processed document IDs from the existing output file.
    """
    processed_ids = set()
    if os.path.exists(output_file):
        with open(output_file, 'r') as f:
            for line in f:
                entry = json.loads(line)
                processed_ids.add(entry["doc_id"])
    return processed_ids

def save_document_embeddings(documents, output_file, batch_size=32):
    """
    Compute and save document embeddings to a file, resuming from the last saved document if interrupted.
    """
    processed_ids = load_processed_ids(output_file)
    total_documents = sum(1 for _ in documents)  # Count total documents for progress bar
    documents = read_documents('data01/FIR-s05-medline.json')  # Reset the generator

    with open(output_file, 'a') as f:  # Open in append mode
        doc_batch = []
        
        for document in tqdm(documents, total=total_documents, desc="Saving Document Embeddings"):
            doc_id = document.get("_id")
            
            # Skip already processed documents
            if doc_id in processed_ids:
                continue
            
            text = prepare_text(document)
            doc_batch.append((doc_id, text))
            
            # Process in batches
            if len(doc_batch) == batch_size:
                ids, texts = zip(*doc_batch)
                embeddings = get_bert_embeddings(texts)
                for doc_id, embedding in zip(ids, embeddings):
                    embedding_data = {"doc_id": doc_id, "embedding": embedding.cpu().tolist()}
                    f.write(json.dumps(embedding_data) + "\n")
                doc_batch.clear()  # Clear the batch
        
        # Process any remaining documents
        if doc_batch:
            ids, texts = zip(*doc_batch)
            embeddings = get_bert_embeddings(texts)
            for doc_id, embedding in zip(ids, embeddings):
                embedding_data = {"doc_id": doc_id, "embedding": embedding.cpu().tolist()}
                f.write(json.dumps(embedding_data) + "\n")

# Usage
#documents = read_documents('data01/FIR-s05-medline.json')
#save_document_embeddings(documents, 'data01/embeddings/pubmed_bert.json', batch_size=320)

Processing Documents:   0%|▍                                                                                                                   | 960/263080 [02:53<13:22:56,  5.44it/s]

In [6]:
import os
import json
import torch
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

def load_document_embeddings(embedding_file):
    document_embeddings = []
    total_embeddings = sum(1 for _ in open(embedding_file, 'r'))  # Count total lines for progress bar

    with open(embedding_file, 'r') as f:
        for line in tqdm(f, total=total_embeddings, desc="Loading Document Embeddings"):
            data = json.loads(line)
            doc_id = data["doc_id"]
            embedding = torch.tensor(data["embedding"])
            document_embeddings.append((doc_id, embedding))
    
    return document_embeddings

def save_query_embedding(qid, query_embedding, cache_file):
    with open(cache_file, 'a') as f:
        json.dump({'qid': qid, 'embedding': query_embedding.cpu().tolist()}, f)
        f.write('\n')  

def load_or_compute_query_embedding(qid, query, cache_file):
    # Load the cache of query embeddings if it exists
    query_embeddings = {}
    if os.path.exists(cache_file):
        with open(cache_file, 'r') as f:
            for line in f:
                data = json.loads(line)
                query_embeddings[data['qid']] = torch.tensor(data['embedding'])
                
    # Return the cached embedding if available
    if qid in query_embeddings:
        return query_embeddings[qid]

    # Compute the embedding if not in cache
    query_embedding = get_bert_embeddings([query])[0]  # Get the embedding
    save_query_embedding(qid, query_embedding, cache_file)  # Save it to the cache
    return query_embedding

def make_trec_run_bert(topics_file_name, run_file_bert_name, embedding_file, run_name="bert", cache_file="data01/embeddings/query_pubmed.json"):
    # Load precomputed document embeddings
    document_embeddings = load_document_embeddings(embedding_file)

    with open(run_file_bert_name, 'w') as run_file:
        with open(topics_file_name, 'r') as test_queries:
            for line in tqdm(test_queries, desc="Processing Queries"):
                qid, query = line.strip().split('\t')

                # Load or compute BERT embedding for the query
                query_embedding = load_or_compute_query_embedding(qid, query, cache_file)

                # Calculate similarity between query and each document
                bert_ranked_candidates = []
                for doc_id, doc_embedding in document_embeddings:
                    similarity = cosine_similarity(query_embedding.detach().cpu().numpy().reshape(1, -1),
                               doc_embedding.numpy().reshape(1, -1))[0][0]
                    bert_ranked_candidates.append((doc_id, similarity))

                # Sort by similarity score
                bert_ranked_candidates.sort(key=lambda x: x[1], reverse=True)

                # Write BERT-only results in TREC format
                for rank, (doc_id, bert_score) in enumerate(bert_ranked_candidates[:1000]):
                    run_file.write(f"{qid} Q0 {doc_id} {rank + 1} {bert_score:.4f} {run_name}\n")

make_trec_run_bert('data01/FIR-s05-training-queries-simple.txt', 'result/bert-pubmed.run', 'data01/embeddings/pubmed_bert.json')


Loading Document Embeddings: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 263080/263080 [01:59<00:00, 2206.32it/s]
Processing Queries: 38it [3:48:06, 360.18s/it] 


## Bert + ES ranking and retrieval

In [ ]:
def make_trec_run_with_bert(es, topics_file_name, run_file_bert_name, index_name="genomics", run_name="es_bert_rerank"):
    with open(run_file_bert_name, 'w') as run_file:
        with open(topics_file_name, 'r') as test_queries:
            for line in test_queries:
                qid, query = line.strip().split('\t')
                
                # Elasticsearch retrieval
                s = elasticsearch_dsl.Search(using=es, index=index_name).query('multi_match', query=query, fields=['TI', 'AB'])
                response = s.execute()

                # Collect candidate documents
                candidates = []
                for hit in response.hits.hits[:100]:
                    doc_id = hit['_source']['PMID']
                    text = f"{hit['_source'].get('TI', '')} {hit['_source'].get('AB', '')}"
                    candidates.append((doc_id, text))

                # BERT re-ranking
                query_embedding = get_bert_embedding(query)
                bert_ranked_candidates = []
                for doc_id, text in candidates:
                    doc_embedding = get_bert_embedding(text)
                    similarity = cosine_similarity(query_embedding.detach().numpy(), doc_embedding.detach().numpy())[0][0]
                    bert_ranked_candidates.append((doc_id, similarity))

                # Sort by BERT similarity score
                bert_ranked_candidates.sort(key=lambda x: x[1], reverse=True)

                # Write BERT re-ranked results in TREC format
                for rank, (doc_id, bert_score) in enumerate(bert_ranked_candidates[:1000]):
                    run_file.write(f"{qid} Q0 {doc_id} {rank+1} {bert_score:.4f} {run_name}\n")

# Run Elasticsearch + BERT re-ranking
make_trec_run_with_bert(
    es,
    'data01/FIR-s05-training-queries-simple.txt',
    'result/run-testresult-es-bert.run',
    index_name='genomics-base',
    run_name='project01-es-bert'
)


## Evaluate Results

In [ ]:
# change indexing in the qrels to index number instead of pmid

In [12]:
import json

def create_pmid_to_index_map(jsonl_file_path):
    pmid_to_index = {}
    index_id = None

    with open(jsonl_file_path, 'r') as file:
        for line in file:
            data = json.loads(line)

            # Detect 'index' lines to get the latest index_id
            if "index" in data and "_id" in data["index"]:
                index_id = data["index"]["_id"]
                continue  # Go to the next line after getting index_id

            # Look for 'PMID' in lines that have data
            if index_id and "PMID" in data:
                pmid = data["PMID"]
                pmid_to_index[pmid] = index_id
                index_id = None  # Reset index_id after using it
                
    return pmid_to_index

def update_qrels_file_with_indices(qrels_file_path, output_qrels_file_path, pmid_to_index):
    with open(qrels_file_path, 'r') as qrels_file, open(output_qrels_file_path, 'w') as updated_qrels_file:
        for line in qrels_file:
            query_id, q0, pmid, relevance = line.strip().split()
            
            index_id = pmid_to_index.get(pmid)
            if index_id:
                updated_qrels_file.write(f"{query_id} {q0} {index_id} {relevance}\n")
            else:
                print(f"Warning: PMID {pmid} not found in the mapping.")

# Usage
pmid_to_index = create_pmid_to_index_map('data01/FIR-s05-medline.json')
update_qrels_file_with_indices('data01/FIR-s05-training-qrels.txt', 'data01/trec-updated.qrels', pmid_to_index)


In [8]:
import sys
import os
module_path = os.path.abspath(os.path.join('performance_evaluation.py'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [9]:
from performance_evaluation import print_trec_eval

In [10]:
print_trec_eval('data01/trec-updated.qrels', 'result/bert-pubmed.run')

Results for result/bert-pubmed.run
mean success_at_1              0.0
mean success_at_5              0.02632
mean success_at_10             0.05263
mean r_precision               0.004138
mean precision_at_1            0.0
mean precision_at_5            0.005263
mean precision_at_10           0.005263
mean precision_at_50           0.004737
mean precision_at_100          0.003421
mean precision_at_recall_00    0.01507
mean precision_at_recall_01    0.01404
mean precision_at_recall_02    0.009414
mean precision_at_recall_03    0.003211
mean precision_at_recall_04    0.002817
mean precision_at_recall_05    0.00276
mean precision_at_recall_06    0.0005251
mean precision_at_recall_07    0.0001671
mean precision_at_recall_08    0.0001671
mean precision_at_recall_09    0.0001671
mean precision_at_recall_10    0.0001671
mean average_precision         0.004072
